## Carregando os dados

In [1]:
import pandas as pd

df = pd.read_csv('../data/churn_tratado.csv')
print(df.shape)
df.head()

(5000, 19)


,Customer_ID,Age,Gender,Subscription_Length,Region,Payment_Method,Support_Tickets_Raised,Satisfaction_Score,Discount_Offered,Last_Activity,Monthly_Spend,Churned,Faixa_Subscription,Faixa_Monthly_Spend,Faixa_Discount,Faixa_Age,Faixa_Last_Activity,Faixa_Tickets,Faixa_Gasto
0,CUST000001,56.0,Male,54,South,PayPal,0,9.0,6.42,319,62.11,1,Maior,Maior,Menor,Mais velhos ainda,Maior,0 a 4 tickets,Acima de 60
1,CUST000002,69.0,Female,21,East,Debit Card,1,2.0,13.77,166,37.27,1,Baixo-médio,Menor,Alto-médio,Mais velhos ainda,Baixo-médio,0 a 4 tickets,30–40
2,CUST000003,46.0,Female,49,East,PayPal,3,8.0,19.91,207,61.82,0,Maior,Maior,Maior,Mais velhos,Alto-médio,0 a 4 tickets,Acima de 60
3,CUST000004,32.0,Male,47,West,Debit Card,3,1.0,13.39,108,40.96,1,Maior,Baixo-médio,Alto-médio,Mais jovens,Baixo-médio,0 a 4 tickets,40–50
4,CUST000005,60.0,Male,6,East,Credit Card,2,6.0,13.18,65,45.97,0,Menor,Baixo-médio,Alto-médio,Mais velhos ainda,Menor,0 a 4 tickets,40–50


## Listando as colunas e decidindo o que entra no modelo

In [2]:
df.dtypes

Customer_ID                object
Age                       float64
Gender                     object
Subscription_Length         int64
Region                     object
Payment_Method             object
Support_Tickets_Raised      int64
Satisfaction_Score        float64
Discount_Offered          float64
Last_Activity               int64
Monthly_Spend             float64
Churned                     int64
Faixa_Subscription         object
Faixa_Monthly_Spend        object
Faixa_Discount             object
Faixa_Age                  object
Faixa_Last_Activity        object
Faixa_Tickets              object
Faixa_Gasto                object
dtype: object

## Separando features (X) e alvo (y)

In [3]:
colunas_numericas = ['Age', 'Subscription_Length', 'Support_Tickets_Raised', 
                      'Satisfaction_Score', 'Discount_Offered', 'Last_Activity', 
                      'Monthly_Spend']

colunas_categoricas = ['Gender', 'Region', 'Payment_Method']

features = colunas_numericas + colunas_categoricas

X = df[features]
y = df['Churned']

print(X.shape)
print(y.value_counts())

(5000, 10)
Churned
0    2760
1    2240
Name: count, dtype: int64


## Dividindo em treino e teste

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)
print()
print('Proporção treino:')
print(y_train.value_counts(normalize=True))
print()
print('Proporção teste:')
print(y_test.value_counts(normalize=True))

Treino: (4000, 10)
Teste: (1000, 10)

Proporção treino:
Churned
0    0.552
1    0.448
Name: proportion, dtype: float64

Proporção teste:
Churned
0    0.552
1    0.448
Name: proportion, dtype: float64


## Converter colunas categóricas em números (encoding)

In [5]:
X_train_encoded = pd.get_dummies(X_train, columns=colunas_categoricas, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=colunas_categoricas, drop_first=True)

print(X_train_encoded.shape)
X_train_encoded.head()

(4000, 13)


,Age,Subscription_Length,Support_Tickets_Raised,Satisfaction_Score,Discount_Offered,Last_Activity,Monthly_Spend,Gender_Male,Region_North,Region_South,Region_West,Payment_Method_Debit Card,Payment_Method_PayPal
100,62.0,24,1,8.0,19.90,294,56.35,True,False,False,False,False,True
2330,57.0,9,2,6.0,14.25,195,42.38,True,False,False,True,True,False
1112,40.0,46,5,6.0,5.33,173,45.90,False,False,False,True,True,False
1644,69.0,38,1,3.0,11.97,47,45.17,True,False,False,True,False,True
345,50.0,56,5,9.0,16.25,155,62.50,False,False,False,False,True,False


## Treinando o primeiro modelo (Regressão Logística)

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Padroniza as variáveis
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# 2. Treina o modelo com os dados padronizados
modelo_lr = LogisticRegression(max_iter=1000, random_state=42)
modelo_lr.fit(X_train_scaled, y_train)

print('Modelo treinado com sucesso')

Modelo treinado com sucesso


## Avaliando o modelo nos dados de teste

In [7]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = modelo_lr.predict(X_test_scaled)

print('Acurácia:', accuracy_score(y_test, y_pred))
print()
print('Matriz de confusão:')
print(confusion_matrix(y_test, y_pred))
print()
print('Relatório completo:')
print(classification_report(y_test, y_pred))

Acurácia: 0.808

Matriz de confusão:
[[466  86]
 [106 342]]

Relatório completo:
              precision    recall  f1-score   support

           0       0.81      0.84      0.83       552
           1       0.80      0.76      0.78       448

    accuracy                           0.81      1000
   macro avg       0.81      0.80      0.81      1000
weighted avg       0.81      0.81      0.81      1000



## Vendo quais variáveis mais influenciam o modelo

In [8]:
import numpy as np

coeficientes = pd.DataFrame({
    'variavel': X_train_encoded.columns,
    'coeficiente': modelo_lr.coef_[0]
})

coeficientes['peso_absoluto'] = coeficientes['coeficiente'].abs()
coeficientes = coeficientes.sort_values('peso_absoluto', ascending=False)

print(coeficientes)

                     variavel  coeficiente  peso_absoluto
6               Monthly_Spend    -1.680548       1.680548
5               Last_Activity     1.305591       1.305591
1         Subscription_Length     0.955850       0.955850
3          Satisfaction_Score    -0.684664       0.684664
4            Discount_Offered     0.368673       0.368673
2      Support_Tickets_Raised     0.337815       0.337815
12      Payment_Method_PayPal     0.139491       0.139491
11  Payment_Method_Debit Card     0.122132       0.122132
8                Region_North     0.090726       0.090726
10                Region_West     0.080104       0.080104
9                Region_South    -0.034699       0.034699
7                 Gender_Male     0.032126       0.032126
0                         Age     0.009093       0.009093
